In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# 1. Menyiapkan daftar kolom (MovieLens memisahkan 19 genre menjadi angka 0 dan 1)
item_cols = ['item_id', 'title', 'release_date', 'video_release_date', 'imdb_url', 'unknown',
             'Action', 'Adventure', 'Animation', 'Children', 'Comedy', 'Crime', 'Documentary',
             'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'Musical', 'Mystery', 'Romance',
             'Sci-Fi', 'Thriller', 'War', 'Western']

# 2. Membaca metadata item (menggunakan encoding latin-1 karena ada teks karakter khusus)
item_path = '../data/raw/ml-100k/u.item'
df_items = pd.read_csv(item_path, sep='|', names=item_cols, encoding='latin-1')

# 3. Kita buang kolom yang tidak relevan untuk rekomendasi fisik (seperti URL)
# Fokus pada ID, judul, dan seluruh kolom genre
genre_columns = item_cols[5:]
df_item_features = df_items[['item_id', 'title'] + genre_columns]

print(f"Total Item: {len(df_item_features)}")
display(df_item_features.head())

Total Item: 1682


,item_id,title,unknown,Action,Adventure,Animation,Children,Comedy,Crime,Documentary,...,Fantasy,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,1,Toy Story (1995),0,0,0,1,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
1,2,GoldenEye (1995),0,1,1,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
2,3,Four Rooms (1995),0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
3,4,Get Shorty (1995),0,1,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
4,5,Copycat (1995),0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,1,0,0


In [2]:
# 1. Mengambil hanya angka genre-nya saja sebagai matriks (1682 baris x 19 genre)
item_feature_matrix = df_item_features[genre_columns].values

# 2. Menghitung Cosine Similarity
# Ini akan membandingkan 1682 film dengan 1682 film lainnya, menghasilkan matriks [1682 x 1682]
item_similarity = cosine_similarity(item_feature_matrix)

# 3. Memasukkan hasil perhitungan ke dalam DataFrame Pandas agar mudah dilacak berdasarkan item_id
df_item_sim = pd.DataFrame(
    item_similarity, 
    index=df_item_features['item_id'], 
    columns=df_item_features['item_id']
)

print("Matriks Kemiripan Antar Item (Cosine Similarity):")
# Tampilan sebagian: Semakin mendekati 1, semakin mirip item kolom dengan item baris
display(df_item_sim.iloc[:5, :5])

Matriks Kemiripan Antar Item (Cosine Similarity):


item_id,1,2,3,4,5
item_id,,,,,
1,1.000000,0.000000,0.00000,0.333333,0.000000
2,0.000000,1.000000,0.57735,0.333333,0.333333
3,0.000000,0.577350,1.00000,0.000000,0.577350
4,0.333333,0.333333,0.00000,1.000000,0.333333
5,0.000000,0.333333,0.57735,0.333333,1.000000


In [3]:
# Memuat riwayat interaksi user
df_train = pd.read_csv('../data/processed/train.csv')

def get_content_based_recommendations(user_id, top_n=5):
    # 1. Filter: Ambil barang yang di-rating positif (misal >= 4) oleh user ini
    user_history = df_train[(df_train['user_id'] == user_id) & (df_train['rating'] >= 4)]
    
    if user_history.empty:
        return {"error": "User tidak memiliki histori positif untuk dipelajari atributnya."}
    
    liked_item_ids = user_history['item_id'].tolist()
    
    # 2. Ambil skor kemiripan dari item yang disukai tersebut terhadap SEMUA item lainnya.
    # Lalu kita rata-ratakan secara vertikal (mean) untuk melihat dominasi profil user.
    sim_scores = df_item_sim.loc[liked_item_ids].mean(axis=0)
    
    # 3. Hapus seluruh item yang sudah pernah berinteraksi dengan user agar tidak direkomendasikan ulang
    all_seen_items = df_train[df_train['user_id'] == user_id]['item_id'].tolist()
    sim_scores = sim_scores.drop(all_seen_items, errors='ignore')
    
    # 4. Urutkan skor dari yang paling mirip (tertinggi)
    top_items = sim_scores.sort_values(ascending=False).head(top_n)
    
    # Format hasil output dengan menempelkan judul filmnya
    result = []
    for item_id, score in top_items.items():
        title = df_item_features[df_item_features['item_id'] == item_id]['title'].values[0]
        result.append({
            "item_id": int(item_id),
            "title": title,
            "similarity_score": round(score, 3)
        })
        
    return {
        "user_id": user_id,
        "recommendation_type": "content_based",
        "recommendations": result
    }

# Mari kita tes modelnya!
print("Rekomendasi Content-Based untuk User 12:")
for rec in get_content_based_recommendations(user_id=12, top_n=5)['recommendations']:
    print(rec)

Rekomendasi Content-Based untuk User 12:
{'item_id': 74, 'title': 'Faster Pussycat! Kill! Kill! (1965)', 'similarity_score': 0.419}
{'item_id': 1424, 'title': 'I Like It Like That (1994)', 'similarity_score': 0.415}
{'item_id': 778, 'title': 'Don Juan DeMarco (1995)', 'similarity_score': 0.415}
{'item_id': 731, 'title': 'Corrina, Corrina (1994)', 'similarity_score': 0.415}
{'item_id': 692, 'title': 'American President, The (1995)', 'similarity_score': 0.415}
